# Chicago traffic assignment

Use AequilibraE 2.0 to find a shortest path, assign morning peak demand, and compare how three groups of drivers respond to different link costs.

## Open the Chicago model

The workshop includes the upgraded model in `data/chicago_sample_model`. Open it to read the network, OMX demand and saved results. The network and demand stay in memory after the project closes.

In [ ]:
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
from aequilibrae import Project
from aequilibrae.matrix import from_dict
from aequilibrae.routing import Network
from aequilibrae.traffic import TrafficAssignment, TrafficClass
from aequilibrae.traffic.vdf import bpr
from IPython.display import display
from lonboard import Map, PathLayer
from shapely.geometry import Point

model_path = Path("data/chicago_sample_model")
if not model_path.is_dir():
    model_path = Path("workshop") / model_path

demand_names = ["car", "heavy_trucks", "medium_trucks"]

with Project.open(model_path) as project:
    links = project.tables.links.data
    modes = project.tables.modes.data
    network = project.road.build_network(
        fields=["free_flow_time", "capacity", "b", "power"], modes=["c"]
    )
    matrix_files = project.matrices.list()
    demand = project.matrices.get_matrix("am.omx", subset=demand_names)
    saved_results = project.results.list()[["table_name", "procedure"]]

display(modes)
links.loc[links["link_id"] == 21615]

## Find a shortest path

Build a graph for road mode `c` using free-flow travel time. Read the physical link IDs and directions in the order travelled from zone 1 to zone 8. A direction of 1 follows the link from A to B, and −1 goes from B to A.

In [ ]:
graph = network.build_graph("c", cost="free_flow_time")
path = graph.compute_path(1, 8)
link_ids, directions = path.get_network_links()

pd.DataFrame({"link_id": link_ids, "direction": directions}).head(10)

## Assign morning peak demand

The AM matrices contain hourly average trips for the morning peak. Cars, heavy trucks and medium trucks share a single traffic class here. They use the same mode, routing costs and passenger car equivalent (PCE) factor of 1.

The assignment builds the class graph and uses the network's `b` and `power` fields for the BPR volume–delay function. The ten-iteration limit keeps the run short. Check the reported relative gap (`rgap`) to see how close it came to the target.

In [ ]:
display(matrix_files[["description"]])
pd.Series({name: demand[name].sum() for name in demand}, name="Hourly trips")

In [ ]:
road_class = TrafficClass(name="road", mode="c", cost="free_flow_time", demand=demand)

assignment = TrafficAssignment(
    network=network,
    classes=[road_class],
    time_field="free_flow_time",
    capacity_field="capacity",
    vdf=bpr,
    name_mapping={"alpha": "b", "beta": "power"},
    algorithm="bfw",
)

results = assignment.execute(max_iterations=10, rgap=1e-6, cores=4)
results[
    ["car_tot", "medium_trucks_tot", "heavy_trucks_tot", "PCE_tot", "VOC_max"]
].head()

### Read the results

The assignment returns one row per physical link, indexed by `link_id`. Load columns use the demand matrix name with `_ab`, `_ba` or `_tot`. The shared results include total PCE flow (`PCE_tot`) and the maximum directional volume-to-capacity ratio (`VOC_max`). The project also includes results from earlier runs.

A blank `alpha` in the final row means that row records a convergence check only.

In [ ]:
display(assignment.report()[["iteration", "rgap", "alpha"]])
display(saved_results)

road_loads = assignment.class_results["road"].get_load_results()
road_loads.head()

### Map flows and congestion

Choose a metric to change the line widths. Each map uses one scale so you can compare the options. Hover over a link to see its values.

In [ ]:
def show_link_results(results, options):
    columns = list(options.values())
    map_links = links[["link_id", "link_type", "geometry"]].join(
        results[columns], on="link_id", validate="one_to_one"
    )
    values = map_links[columns].fillna(0)
    scale = max(1, values.max().max())
    widths = 14 * values / scale

    metric = widgets.Dropdown(options=options, description="Show")
    layer = PathLayer.from_geopandas(
        map_links,
        get_width=widths[metric.value].to_numpy(dtype="float32"),
        get_color=[22, 93, 180],
        width_units="pixels",
    )

    def update_width(change):
        layer.get_width = widths[change["new"]].to_numpy(dtype="float32")

    metric.observe(update_width, names="value")
    traffic_map = Map(layers=[layer], basemap=None, height="540px", show_tooltip=True)
    return widgets.VBox([metric, traffic_map])


show_link_results(
    results,
    {
        "All traffic (PCE)": "PCE_tot",
        "Cars": "car_tot",
        "Medium trucks": "medium_trucks_tot",
        "Heavy trucks": "heavy_trucks_tot",
    },
)

In [ ]:
show_link_results(
    results,
    {"Maximum ratio": "VOC_max", "A to B": "VOC_AB", "B to A": "VOC_BA"},
)

## Compare three traffic classes

Split car demand evenly across three classes. All three share road capacity, and each class adds a different fixed penalty to roads inside the same area near central Chicago. The penalties use the same time units as `free_flow_time`. Centroid connectors are outside the penalty calculation.

Use real link IDs to add an area flag to a new in-memory network. Then set the fixed cost multiplier on each class to give it a low, medium or high penalty on those roads.

In [ ]:
penalty_area = Point(-87.63019, 41.81387).buffer(0.1)
affected_links = links.loc[
    links.geometry.within(penalty_area) & (links["link_type"] != "centroid_connector"),
    "link_id",
]
penalty_links = network.links.assign(
    area_penalty=network.links["link_id"].isin(affected_links).astype(float)
)
penalty_network = Network(
    penalty_links,
    modes=network.modes,
    centroids=network.centroids,
    turn_restrictions=network.turn_restrictions,
    allow_uturns=network.allow_uturns,
    lonlat=network.lonlat,
)

penalties = {"low": 0.0, "medium": 0.25, "high": 1.0}
demand_per_class = demand["car"] / len(penalties)
classes = [
    TrafficClass(
        name=name,
        mode="c",
        cost="free_flow_time",
        demand=from_dict(
            {name: demand_per_class}, index=demand.index, index_name=demand.index_name
        ),
        fixed_cost_field="area_penalty",
        fixed_cost_multiplier=penalty,
    )
    for name, penalty in penalties.items()
]

In [ ]:
multi_assignment = TrafficAssignment(
    network=penalty_network,
    classes=classes,
    time_field="free_flow_time",
    capacity_field="capacity",
    vdf=bpr,
    name_mapping={"alpha": "b", "beta": "power"},
    algorithm="bfw",
)

multi_results = multi_assignment.execute(max_iterations=10, rgap=1e-6, cores=4)
display(multi_assignment.report()[["iteration", "rgap", "alpha"]])
multi_results[["low_tot", "medium_tot", "high_tot"]].head()

Summing link flows inside the area counts a trip once for every affected link it uses. Compare the sums and switch between the three classes on the map to see where routes change.

In [ ]:
class_columns = ["low_tot", "medium_tot", "high_tot"]
multi_results.loc[affected_links, class_columns].sum().rename(
    "Vehicles per hour across affected links"
)

In [ ]:
show_link_results(
    multi_results,
    {
        "Low penalty": "low_tot",
        "Medium penalty": "medium_tot",
        "High penalty": "high_tot",
    },
)